# Notebook 10 — DeepFM Fine Ranking (PyTorch)

**Objective:** learn feature interactions using a lightweight DeepFM ranker and compare it fairly with Hybrid RRF and LightGBM on the **same August user-disjoint holdout**. After selecting the epoch on August holdout, refit on all August users and report September test once.

**Data limitation:** RetailRocket contains observed interactions but no impression log. The binary targets use **sampled implicit negatives** and **do not represent calibrated CTR/CVR**. The candidate pool remains fixed at Notebook 08 Top-100. DeepFM here is an experimental candidate re-ranker, not a proven production-grade ads model.

**Important:** We have already inspected September test results in prior notebooks; avoid tuning the architecture based on test performance. Use August holdout for decisions. Notebook 11 will add richer time-safe features.

## 1. Load dependencies and Notebook 08 candidate exports

In [1]:
# pip install torch scikit-learn pandas numpy lightgbm
from pathlib import Path
import json, time, pickle, random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED=42
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
torch.set_num_threads(min(4,torch.get_num_threads()))
ROOT=Path.cwd().resolve()
if not (ROOT/'data'/'processed').is_dir(): ROOT=ROOT.parent
assert (ROOT/'data'/'processed').is_dir(), 'Run notebook from project root or notebooks/'
PROC=ROOT/'data'/'processed';REPORTS=ROOT/'reports';REPORTS.mkdir(exist_ok=True)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Project:',ROOT,'Device:',DEVICE)
CUT={'val':pd.Timestamp('2015-08-01',tz='UTC'),'test':pd.Timestamp('2015-09-01',tz='UTC')}
raw={}
for split,stem,hfile in [('val','validation','train_history.csv'),('test','test','test_history.csv')]:
    cand=pd.read_csv(PROC/f'notebook08_{split}_hybrid_candidates.csv')
    hist=pd.read_csv(PROC/hfile,usecols=['timestamp','visitorid','itemid','event'])
    labels=pd.read_csv(PROC/f'{stem}_view_labels_warm_users.csv',usecols=['visitorid','itemid']).drop_duplicates()
    assert hist.timestamp.max()<int(CUT[split].timestamp()*1000)
    assert not cand[['visitorid','itemid']].duplicated().any()
    assert cand.groupby('visitorid').size().eq(100).all()
    assert set(cand.visitorid.unique())==set(labels.visitorid.unique())
    raw[split]=(cand,hist,labels)
    print(split,'candidates:',len(cand),'labels:',len(labels))

Project: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks Device: cuda
val candidates: 1332900 labels: 28634
test candidates: 1513700 labels: 31876


## 2. Recreate Notebook 09 features exactly

The following 15 numerical features are constructed only from events before each cutoff. For comparability we use Notebook 09's same user-disjoint split and negative-sampling policy. **No user ID or item ID embeddings are trained:** memorizing sparse IDs with only ~1.5K sampled positive training rows would be fragile; this DeepFM models interactions among the existing feature fields instead.

In [2]:
FEATURES=['retrieval_rank','rrf_inverse_rank','from_itemcf','from_faiss_fallback','from_popularity',
          'source_count','log_item_views','log_item_unique_viewers','log_user_views',
          'log_user_unique_items','user_last_event_days','item_last_view_days',
          'item_view_share','log_item_carts','log_item_purchases']

def make_features(candidates,history,cutoff):
    views=history.loc[history.event.eq('view'),['visitorid','itemid','timestamp']]
    stats=views.groupby('itemid',sort=False).agg(item_views=('visitorid','size'),
        item_unique_viewers=('visitorid','nunique'),item_last_view=('timestamp','max'))
    users=views.groupby('visitorid',sort=False).agg(user_views=('itemid','size'),
        user_unique_items=('itemid','nunique'),user_last_event=('timestamp','max'))
    other=history.loc[history.event.isin(['addtocart','transaction']),['itemid','event']]
    if len(other):
        counts=other.groupby(['itemid','event']).size().unstack(fill_value=0)
        stats['item_carts']=counts['addtocart'] if 'addtocart' in counts else 0
        stats['item_purchases']=counts['transaction'] if 'transaction' in counts else 0
    else:stats['item_carts']=0;stats['item_purchases']=0
    stats[['item_carts','item_purchases']]=stats[['item_carts','item_purchases']].fillna(0)
    d=candidates.merge(stats,left_on='itemid',right_index=True,how='left',validate='many_to_one')
    d=d.merge(users,left_on='visitorid',right_index=True,how='left',validate='many_to_one')
    ms=int(cutoff.timestamp()*1000)
    d['retrieval_rank']=d['rank'].astype('float32')
    d['rrf_inverse_rank']=(1.0/(60+d['rank'])).astype('float32')
    d['source_count']=d[['from_itemcf','from_faiss_fallback','from_popularity']].sum(axis=1)
    for src,dst in [('item_views','log_item_views'),('item_unique_viewers','log_item_unique_viewers'),
                    ('user_views','log_user_views'),('user_unique_items','log_user_unique_items'),
                    ('item_carts','log_item_carts'),('item_purchases','log_item_purchases')]:
        d[dst]=np.log1p(d[src].fillna(0)).astype('float32')
    d['user_last_event_days']=((ms-d.user_last_event)/86400000).clip(lower=0).fillna(999).astype('float32')
    d['item_last_view_days']=((ms-d.item_last_view)/86400000).clip(lower=0).fillna(999).astype('float32')
    d['item_view_share']=(d.item_views.fillna(0)/max(1,len(views))).astype('float32')
    for name in FEATURES:d[name]=d[name].fillna(0).astype('float32')
    return d[['visitorid','itemid','rank']+FEATURES].copy()

def attach_labels(df,labels):
    pos=labels[['visitorid','itemid']].drop_duplicates().assign(label=np.int8(1))
    out=df.merge(pos,on=['visitorid','itemid'],how='left',validate='one_to_one')
    out['label']=out.label.fillna(0).astype('int8')
    return out

frames={}
for split in ('val','test'):
    c,h,l=raw[split]
    frames[split]=attach_labels(make_features(c,h,CUT[split]),l)
    assert frames[split][FEATURES].notna().all().all()
    print(split,frames[split].shape,'positive candidates:',int(frames[split].label.sum()))


val (1332900, 19) positive candidates: 1888
test (1513700, 19) positive candidates: 1829


## 3. Deterministic user-disjoint August holdout and negative sampling

In [3]:
val=frames['val'];test=frames['test']
all_users=np.sort(val.visitorid.unique())
train_u,holdout_u=train_test_split(all_users,test_size=.20,random_state=SEED)
train_u=set(map(int,train_u));holdout_u=set(map(int,holdout_u))
assert not train_u.intersection(holdout_u)

def sample_train_rows(frame,user_ids,neg_per_user=20,seed=SEED):
    sub=frame.loc[frame.visitorid.isin(user_ids)].copy()
    positives=sub[sub.label.eq(1)]
    negatives=sub[sub.label.eq(0)]
    negatives=(negatives.assign(_random=np.random.default_rng(seed).random(len(negatives)))
               .sort_values(['visitorid','_random'],kind='mergesort')
               .groupby('visitorid',sort=False).head(neg_per_user).drop(columns='_random'))
    return pd.concat([positives,negatives],ignore_index=True).sort_values(['visitorid','rank'],kind='mergesort').reset_index(drop=True)

train_df=sample_train_rows(val,train_u)
holdout_df=val[val.visitorid.isin(holdout_u)].sort_values(['visitorid','rank']).reset_index(drop=True)
all_aug=sample_train_rows(val,set(map(int,all_users)))
assert train_df.label.sum()>0 and holdout_df.label.sum()>0
print('Train:',train_df.shape,'positives:',int(train_df.label.sum()))
print('Holdout:',holdout_df.shape,'positives:',int(holdout_df.label.sum()))
print('Full-Aug:',all_aug.shape,'positives:',int(all_aug.label.sum()))

Train: (214772, 19) positives: 1512
Holdout: (266600, 19) positives: 376
Full-Aug: (268468, 19) positives: 1888


## 4. Define a numerical-feature DeepFM

DeepFM combines a linear term, pairwise Factorization Machine interactions, and an MLP. This version uses **15 numerical feature fields**, with one learnable vector per field; it does not embed user/item identities. Fit the scaler on training rows only. Class imbalance is handled with a positive class weight, but the resulting score is **not calibrated**.

In [4]:
class NumericDeepFM(nn.Module):
    def __init__(self,n_fields,embed_dim=8,hidden=(64,32),dropout=.2):
        super().__init__()
        self.linear=nn.Linear(n_fields,1)
        self.fm_vectors=nn.Parameter(torch.randn(n_fields,embed_dim)*.02)
        layers=[];d=n_fields*embed_dim
        for width in hidden:
            layers.extend([nn.Linear(d,width),nn.ReLU(),nn.Dropout(dropout)])
            d=width
        layers.append(nn.Linear(d,1))
        self.mlp=nn.Sequential(*layers)
    def forward(self,x):
        embeddings=x.unsqueeze(-1)*self.fm_vectors.unsqueeze(0)
        summed=embeddings.sum(dim=1)
        fm=.5*(summed.square()-(embeddings.square()).sum(dim=1)).sum(dim=1,keepdim=True)
        return (self.linear(x)+fm+self.mlp(embeddings.flatten(start_dim=1))).squeeze(-1)

def prepare_scaler(frame):
    scaler=StandardScaler()
    scaler.fit(frame[FEATURES].to_numpy(dtype=np.float32))
    return scaler

def transform(frame,scaler):
    x=scaler.transform(frame[FEATURES].to_numpy(dtype=np.float32))
    return np.clip(x,-8,8).astype('float32')

def train_epoch(model,loader,optimizer,criterion):
    model.train();total=0.;seen=0
    for xb,yb in loader:
        xb=xb.to(DEVICE);yb=yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss=criterion(model(xb),yb)
        loss.backward();nn.utils.clip_grad_norm_(model.parameters(),5.0);optimizer.step()
        total+=float(loss.item())*len(xb);seen+=len(xb)
    return total/max(1,seen)

@torch.no_grad()
def score_model(model,frame,scaler,batch_size=16384):
    model.eval();x=transform(frame,scaler);scores=[]
    for start in range(0,len(x),batch_size):
        xb=torch.from_numpy(x[start:start+batch_size]).to(DEVICE)
        scores.append(model(xb).cpu().numpy())
    return np.concatenate(scores)

scaler=prepare_scaler(train_df)
train_x=transform(train_df,scaler)
train_y=train_df.label.to_numpy(dtype=np.float32)
loader=DataLoader(TensorDataset(torch.from_numpy(train_x),torch.from_numpy(train_y)),
                  batch_size=4096,shuffle=True,num_workers=0)
pos_weight=min(100.,float((train_y==0).sum()/max(1,(train_y==1).sum())))
model=NumericDeepFM(len(FEATURES)).to(DEVICE)
optimizer=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=1e-4)
criterion=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight,device=DEVICE))
print('DeepFM parameters:',sum(p.numel() for p in model.parameters()),'positive weight:',round(pos_weight,2))

DeepFM parameters: 9993 positive weight: 100.0


## 5. Rank metrics and epoch selection on August holdout only

In [5]:
def rank_candidates(frame,scores):
    d=frame[['visitorid','itemid','rank']].copy()
    d['score']=np.asarray(scores,dtype=np.float32)
    d=d.sort_values(['visitorid','score','rank','itemid'],ascending=[True,False,True,True],kind='mergesort')
    d['new_rank']=d.groupby('visitorid',sort=False).cumcount()+1
    return d

def evaluate_ranks(frame,labels,rank_column,k_values=(10,20,50,100)):
    truth=labels.groupby('visitorid').itemid.agg(lambda x:set(map(int,x))).to_dict()
    f=frame.sort_values(['visitorid',rank_column],kind='mergesort')
    recs=f.groupby('visitorid',sort=False).itemid.agg(list).to_dict()
    assert set(recs)==set(truth)
    denominator=sum(map(len,truth.values()))
    total_hits=sum(len(set(items)&truth[int(u)]) for u,items in recs.items())
    rows=[]
    for k in k_values:
        found=0;hit_users=0;macro=[]
        for u,items in recs.items():
            hits=len(set(items[:k])&truth[int(u)])
            found+=hits;hit_users+=int(hits>0);macro.append(hits/len(truth[int(u)]))
        rows.append(dict(K=k,Recall_micro=found/denominator,Recall_macro=float(np.mean(macro)),
                         HitRate=hit_users/len(truth),CandidateConditionedRecall=found/total_hits if total_hits else np.nan,hits=found))
    return pd.DataFrame(rows)

holdout_labels=raw['val'][2].loc[raw['val'][2].visitorid.isin(holdout_u)]
base_holdout=evaluate_ranks(holdout_df,holdout_labels,'rank')
EPOCHS=8;history=[];best_epoch=1;best_recall=-1.;best_state=None
for epoch in range(1,EPOCHS+1):
    loss=train_epoch(model,loader,optimizer,criterion)
    scored=rank_candidates(holdout_df,score_model(model,holdout_df,scaler))
    metrics=evaluate_ranks(scored,holdout_labels,'new_rank')
    r10=float(metrics.loc[metrics.K.eq(10),'Recall_micro'].iloc[0])
    history.append(dict(epoch=epoch,loss=loss,holdout_recall_at_10=r10))
    print(f'Epoch {epoch:02d}: loss={loss:.4f} holdout Recall@10={r10:.5f}')
    if r10>best_recall:
        best_recall=r10;best_epoch=epoch
        best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
model.load_state_dict(best_state)
holdout_ranked=rank_candidates(holdout_df,score_model(model,holdout_df,scaler))
new_holdout=evaluate_ranks(holdout_ranked,holdout_labels,'new_rank')
holdout_comparison=base_holdout.merge(new_holdout,on='K',suffixes=('_hybrid','_deepfm'))
print('Selected epoch:',best_epoch)
display(holdout_comparison)

Epoch 01: loss=1.1277 holdout Recall@10=0.03035
Epoch 02: loss=0.8622 holdout Recall@10=0.03323
Epoch 03: loss=0.8197 holdout Recall@10=0.03362
Epoch 04: loss=0.8133 holdout Recall@10=0.03342
Epoch 05: loss=0.8047 holdout Recall@10=0.03381
Epoch 06: loss=0.8020 holdout Recall@10=0.03400
Epoch 07: loss=0.7913 holdout Recall@10=0.03419
Epoch 08: loss=0.7940 holdout Recall@10=0.03400
Selected epoch: 7


,K,Recall_micro_hybrid,Recall_macro_hybrid,HitRate_hybrid,CandidateConditionedRecall_hybrid,hits_hybrid,Recall_micro_deepfm,Recall_macro_deepfm,HitRate_deepfm,CandidateConditionedRecall_deepfm,hits_deepfm
0,10,0.025740,0.026256,0.045011,0.356383,134,0.034191,0.034244,0.057389,0.473404,178
1,20,0.038033,0.038149,0.063016,0.526596,198,0.045524,0.044656,0.073893,0.630319,237
2,50,0.054552,0.051330,0.081770,0.755319,284,0.062044,0.057446,0.090023,0.859043,323
3,100,0.072224,0.065565,0.100525,1.000000,376,0.072224,0.065565,0.100525,1.000000,376


## 6. Compare with Notebook 09 LightGBM on identical August holdout

The previous Notebook 09 holdout split uses the same seed and sklearn split. We verify Hybrid metrics match before comparing; otherwise stop because evaluation protocols differ.

In [6]:
lgb_path=REPORTS/'notebook09_val_holdout_metrics.csv'
if lgb_path.exists():
    lgb_metrics=pd.read_csv(lgb_path)
    for k in (10,20,50,100):
        a=holdout_comparison.loc[holdout_comparison.K.eq(k),'Recall_micro_hybrid'].iloc[0]
        b=lgb_metrics.loc[lgb_metrics.K.eq(k),'Recall_micro_hybrid'].iloc[0]
        assert np.isclose(a,b,atol=1e-10),f'Notebook 09 protocol mismatch at K={k}: {a} vs {b}'
    table=holdout_comparison[['K','Recall_micro_hybrid','Recall_micro_deepfm']].merge(
        lgb_metrics[['K','Recall_micro_lightgbm']],on='K')
    display(table)
else:
    print('LightGBM metrics file not found; DeepFM vs Hybrid comparison shown above.')

,K,Recall_micro_hybrid,Recall_micro_deepfm,Recall_micro_lightgbm
0,10,0.025740,0.034191,0.032270
1,20,0.038033,0.045524,0.046101
2,50,0.054552,0.062044,0.061468
3,100,0.072224,0.072224,0.072224


## 7. Refit on all August users; evaluate on September without tuning

We retrain from scratch for exactly the chosen number of epochs, using August labels and August-only features. September features are built from pre-September history. Do not select a new epoch based on September outcomes.

In [7]:
torch.manual_seed(SEED)
full_scaler=prepare_scaler(all_aug)
full_x=transform(all_aug,full_scaler)
full_y=all_aug.label.to_numpy(dtype=np.float32)
full_loader=DataLoader(TensorDataset(torch.from_numpy(full_x),torch.from_numpy(full_y)),
                       batch_size=4096,shuffle=True,num_workers=0)
full_weight=min(100.,float((full_y==0).sum()/max(1,(full_y==1).sum())))
final_model=NumericDeepFM(len(FEATURES)).to(DEVICE)
full_opt=torch.optim.AdamW(final_model.parameters(),lr=.001,weight_decay=1e-4)
full_loss=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(full_weight,device=DEVICE))
for epoch in range(1,best_epoch+1):
    print('Full-Aug epoch',epoch,'loss',round(train_epoch(final_model,full_loader,full_opt,full_loss),5))
t0=time.perf_counter()
test_scores=score_model(final_model,test,full_scaler)
scoring_seconds=time.perf_counter()-t0
test_ranked=rank_candidates(test,test_scores)
base_test=evaluate_ranks(test,raw['test'][2],'rank')
new_test=evaluate_ranks(test_ranked,raw['test'][2],'new_rank')
test_comparison=base_test.merge(new_test,on='K',suffixes=('_hybrid','_deepfm'))
assert np.isclose(test_comparison.iloc[-1].Recall_micro_hybrid,test_comparison.iloc[-1].Recall_micro_deepfm)
print('Test scoring seconds:',round(scoring_seconds,3))
display(test_comparison)
lgb_test_path=REPORTS/'notebook09_test_metrics.csv'
if lgb_test_path.exists():
    lgb_test=pd.read_csv(lgb_test_path)
    assert np.allclose(test_comparison.Recall_micro_hybrid,lgb_test.Recall_micro_hybrid)
    display(test_comparison[['K','Recall_micro_hybrid','Recall_micro_deepfm']].merge(
        lgb_test[['K','Recall_micro_lightgbm']],on='K'))

Full-Aug epoch 1 loss 1.08942
Full-Aug epoch 2 loss 0.83847
Full-Aug epoch 3 loss 0.81687
Full-Aug epoch 4 loss 0.80668
Full-Aug epoch 5 loss 0.80208
Full-Aug epoch 6 loss 0.79642
Full-Aug epoch 7 loss 0.79007
Test scoring seconds: 0.747


,K,Recall_micro_hybrid,Recall_macro_hybrid,HitRate_hybrid,CandidateConditionedRecall_hybrid,hits_hybrid,Recall_micro_deepfm,Recall_macro_deepfm,HitRate_deepfm,CandidateConditionedRecall_deepfm,hits_deepfm
0,10,0.020768,0.024067,0.037590,0.361946,662,0.027105,0.030666,0.048490,0.472389,864
1,20,0.029427,0.033581,0.051133,0.512849,938,0.037740,0.042260,0.064346,0.657736,1203
2,50,0.045018,0.049997,0.073198,0.784582,1435,0.050540,0.055258,0.080729,0.880809,1611
3,100,0.057379,0.061646,0.088855,1.000000,1829,0.057379,0.061646,0.088855,1.000000,1829


,K,Recall_micro_hybrid,Recall_micro_deepfm,Recall_micro_lightgbm
0,10,0.020768,0.027105,0.026383
1,20,0.029427,0.037740,0.036767
2,50,0.045018,0.050540,0.049881
3,100,0.057379,0.057379,0.057379


## 8. Export models, candidates and metrics

Saved scores are ranking scores, **not** calibrated probabilities. No future labels are written to the candidate export. Store feature ordering and scaler alongside model weights for later serving.

In [8]:
model_path=REPORTS/'notebook10_deepfm_model.pt'
torch.save({'state_dict':{k:v.cpu() for k,v in final_model.state_dict().items()},
            'features':FEATURES,'n_fields':len(FEATURES),'embed_dim':8,
            'hidden':[64,32],'dropout':.2,'best_epoch':best_epoch,
            'scaler_mean':full_scaler.mean_.tolist(),'scaler_scale':full_scaler.scale_.tolist(),
            'training_cutoff':'2015-08-01'},model_path)
holdout_ranked.to_csv(PROC/'notebook10_val_holdout_scored_candidates.csv',index=False)
test_ranked.to_csv(PROC/'notebook10_test_scored_candidates.csv',index=False)
holdout_comparison.to_csv(REPORTS/'notebook10_val_holdout_metrics.csv',index=False)
test_comparison.to_csv(REPORTS/'notebook10_test_metrics.csv',index=False)
summary={'method':'numerical-feature DeepFM sampled implicit negative re-ranker',
         'protocol':'August user-disjoint holdout; full-Aug refit; September test',
         'caveats':['No impression logs; implicit negatives are not confirmed negatives',
                    'Scores are not calibrated CTR/CVR','Fixed Notebook 08 Top-100 candidate pool'],
         'features':FEATURES,'best_epoch':best_epoch,'epochs_tried':EPOCHS,
         'train_rows':len(train_df),'train_positives':int(train_df.label.sum()),
         'validation_holdout_users':len(holdout_u),
         'validation_holdout':holdout_comparison.to_dict(orient='records'),
         'test':test_comparison.to_dict(orient='records'),
         'test_scoring_seconds':scoring_seconds,'epoch_history':history}
report_path=REPORTS/'notebook10_deepfm_fine_ranking.json'
report_path.write_text(json.dumps(summary,indent=2),encoding='utf-8')
assert not any('label' in x for x in FEATURES)
assert not test_ranked[['visitorid','itemid']].duplicated().any()
assert test_ranked.groupby('visitorid').size().eq(100).all()
assert test_ranked.groupby('visitorid').new_rank.max().eq(100).all()
print('Saved:',report_path)
print('Saved:',model_path)
print('PASS: historical features, user-disjoint holdout, fixed candidate pool, DeepFM exports')

Saved: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\reports\notebook10_deepfm_fine_ranking.json
Saved: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\reports\notebook10_deepfm_model.pt
PASS: historical features, user-disjoint holdout, fixed candidate pool, DeepFM exports


## Interpretation / Next

- Compare DeepFM with LightGBM **on the same August holdout**, especially Recall@10 and HitRate@10. DeepFM is not guaranteed to win with sparse implicit positives.
- Keep the better model according to the holdout; use September only for final reporting, not repeated tuning.
- **Notebook 11:** Ranking Feature Engineering, richer user–item interactions and temporal features without leakage.